# ✅ STEP-BY-STEP CUSTOMER CHURN PREDICTION PROJECT

This notebook predicts whether customers will leave a business using machine learning.

In [52]:
# 🔹 Step 1: Import Required Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [55]:
df = pd.read_csv("BankChurners.csv")

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

Dataset Shape: (10127, 23)

First few rows:
   CLIENTNUM     Attrition_Flag  Customer_Age Gender  Dependent_count  \
0  768805383  Existing Customer            45      M                3   
1  818770008  Existing Customer            49      F                5   
2  713982108  Existing Customer            51      M                3   
3  769911858  Existing Customer            40      F                4   
4  709106358  Existing Customer            40      M                3   

  Education_Level Marital_Status Income_Category Card_Category  \
0     High School        Married     $60K - $80K          Blue   
1        Graduate         Single  Less than $40K          Blue   
2        Graduate        Married    $80K - $120K          Blue   
3     High School        Unknown  Less than $40K          Blue   
4      Uneducated        Married     $60K - $80K          Blue   

   Months_on_book  ...  Credit_Limit  Total_Revolving_Bal  Avg_Open_To_Buy  \
0              39  ...       12691.0      

In [43]:
# 🔹 Step 3: Data Cleaning

# 3.1 Remove Unnecessary Columns
df = df.drop("CLIENTNUM", axis=1)

# 3.2 Convert Target Variable
df["Attrition_Flag"] = df["Attrition_Flag"].map({
    "Attrited Customer": 1,
    "Existing Customer": 0
})

# 3.3 Handle Missing Values
df = df.dropna()

# 3.4 Convert Categorical Data to Numeric
df = pd.get_dummies(df, drop_first=True)

print("✓ Data Cleaning Complete")
print("Cleaned Data Shape:", df.shape)

✓ Data Cleaning Complete
Cleaned Data Shape: (10127, 35)


In [44]:
# 🔹 Step 4: Define Features and Target

X = df.drop("Attrition_Flag", axis=1)
y = df["Attrition_Flag"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (10127, 34)
Target shape: (10127,)


In [45]:
# 🔹 Step 5: Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8101
Testing samples: 2026


In [46]:
# 🔹 Step 6: Feature Scaling (For Logistic Regression)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled successfully")

✓ Features scaled successfully


In [47]:
# 🔹 Step 7: Train Logistic Regression Model

log_model = LogisticRegression(max_iter=1000, class_weight="balanced")
log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)

print("\nLogistic Regression Results")
print("Accuracy:", round(accuracy_score(y_test, y_pred_log), 4))
print(classification_report(y_test, y_pred_log))


Logistic Regression Results
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1701
           1       1.00      1.00      1.00       325

    accuracy                           1.00      2026
   macro avg       1.00      1.00      1.00      2026
weighted avg       1.00      1.00      1.00      2026



In [48]:
# 🔹 Step 8: Train Random Forest Model

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("\nRandom Forest Results")
print("Accuracy:", round(accuracy_score(y_test, y_pred_rf), 4))
print(classification_report(y_test, y_pred_rf))


Random Forest Results
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1701
           1       1.00      1.00      1.00       325

    accuracy                           1.00      2026
   macro avg       1.00      1.00      1.00      2026
weighted avg       1.00      1.00      1.00      2026



In [49]:
# 🔹 Step 9: Model Comparison and Selection

acc_log = accuracy_score(y_test, y_pred_log)
acc_rf = accuracy_score(y_test, y_pred_rf)

if acc_rf > acc_log:
    final_model = rf_model
    model_name = "Random Forest"
else:
    final_model = log_model
    model_name = "Logistic Regression"

print("\nBest Model:", model_name)
print("Logistic Regression Accuracy:", round(acc_log, 4))
print("Random Forest Accuracy:", round(acc_rf, 4))


Best Model: Logistic Regression
Logistic Regression Accuracy: 1.0
Random Forest Accuracy: 1.0


In [50]:
# 🔹 Step 10: Feature Importance (Model Insight)

importances = pd.Series(rf_model.feature_importances_, index=X.columns)

print("\nTop 5 Important Features:")
print(importances.sort_values(ascending=False).head(5))


Top 5 Important Features:
Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2    0.412186
Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1    0.371343
Total_Trans_Ct                                                                                                                        0.044993
Total_Trans_Amt                                                                                                                       0.036931
Total_Revolving_Bal                                                                                                                   0.034193
dtype: float64


In [57]:
# 🔹 Step 13: Two Example Predictions Based on the Models

print("=" * 80)
print("TWO EXAMPLE CUSTOMER PREDICTIONS")
print("=" * 80)

# Helper function to predict one customer row

def predict_customer(row):
    sample_row = row.to_frame().T
    if model_name == "Logistic Regression":
        sample_row = scaler.transform(sample_row)
    prediction_value = final_model.predict(sample_row)[0]
    return prediction_value

stay_idx = None
leave_idx = None

for idx in X_test.index:
    pred_value = predict_customer(X_test.loc[idx])
    if pred_value == 0 and stay_idx is None:
        stay_idx = idx
    elif pred_value == 1 and leave_idx is None:
        leave_idx = idx
    if stay_idx is not None and leave_idx is not None:
        break

print("\nExample 1: Customer likely to STAY")
print("-" * 80)
if stay_idx is not None:
    actual_value = y_test.loc[stay_idx]
    print("Row index:", stay_idx)
    print("Model prediction: Customer will Stay")
    print("Actual status:", "Customer Stayed" if actual_value == 0 else "Customer Left")
else:
    print("No stay example found in the test set.")

print("\nExample 2: Customer likely to LEAVE")
print("-" * 80)
if leave_idx is not None:
    actual_value = y_test.loc[leave_idx]
    print("Row index:", leave_idx)
    print("Model prediction: Customer will Leave")
    print("Actual status:", "Customer Left" if actual_value == 1 else "Customer Stayed")
else:
    print("No leave example found in the test set.")

TWO EXAMPLE CUSTOMER PREDICTIONS

Example 1: Customer likely to STAY
--------------------------------------------------------------------------------
Row index: 2919
Model prediction: Customer will Stay
Actual status: Customer Stayed

Example 2: Customer likely to LEAVE
--------------------------------------------------------------------------------
Row index: 8623
Model prediction: Customer will Leave
Actual status: Customer Left


In [61]:
# 🔹 Step 14: Raw Customer Examples With Yes/No Probabilities

print("=" * 80)
print("RAW CUSTOMER EXAMPLES")
print("=" * 80)

# Match the names used in your example
log_reg = log_model
random_forest = rf_model
final_model_name = model_name

# Keep a copy of the cleaned training columns so new customers can be aligned correctly
target_col = "Churn" if "Churn" in df.columns else "Attrition_Flag"
feature_columns = X.columns

# Helper: convert one raw customer row into the model-ready feature format
def prepare_customer(raw_customer):
    row_df = pd.DataFrame([raw_customer]).copy()

    if "clientnum" in row_df.columns:
        row_df = row_df.drop("clientnum", axis=1)
    if "CLIENTNUM" in row_df.columns:
        row_df = row_df.drop("CLIENTNUM", axis=1)

    if "Attrition_Flag" in row_df.columns:
        row_df = row_df.drop("Attrition_Flag", axis=1)

    if "TotalCharges" in row_df.columns:
        row_df["TotalCharges"] = pd.to_numeric(row_df["TotalCharges"], errors="coerce")

    if "tenure" in row_df.columns and "TotalCharges" in row_df.columns:
        row_df["AvgMonthlySpend"] = (row_df["TotalCharges"] / row_df["tenure"].replace(0, 1)).round(2)
    if "tenure" in row_df.columns:
        row_df["IsNewCustomer"] = (row_df["tenure"] <= 12).astype(int)
    if "PhoneService" in row_df.columns:
        row_df["ServiceCount"] = (
            (row_df["PhoneService"] == "Yes").astype(int)
            + (row_df.get("InternetService", "No") != "No").astype(int)
            + (row_df.get("OnlineSecurity", "No") == "Yes").astype(int)
            + (row_df.get("OnlineBackup", "No") == "Yes").astype(int)
            + (row_df.get("DeviceProtection", "No") == "Yes").astype(int)
            + (row_df.get("TechSupport", "No") == "Yes").astype(int)
            + (row_df.get("StreamingTV", "No") == "Yes").astype(int)
            + (row_df.get("StreamingMovies", "No") == "Yes").astype(int)
        )

    row_df = pd.get_dummies(row_df, drop_first=True)
    row_df = row_df.reindex(columns=feature_columns, fill_value=0)
    return row_df

# Helper: print the prediction in the same style as your example
def show_prediction(title, raw_customer):
    print(f"\n{title}")
    print("-" * 80)
    customer_input = pd.DataFrame([raw_customer])
    print("New customer input:")
    print(customer_input)

    model_input = prepare_customer(raw_customer)
    lr_prob = log_reg.predict_proba(scaler.transform(model_input))[0, 1]
    rf_prob = random_forest.predict_proba(model_input)[0, 1]

    lr_label = "Yes" if lr_prob >= 0.5 else "No"
    rf_label = "Yes" if rf_prob >= 0.5 else "No"

    print("\nPrediction output:")
    print(f"Logistic Regression -> Churn: {lr_label}, Probability: {lr_prob:.2%}")
    print(f"Random Forest       -> Churn: {rf_label}, Probability: {rf_prob:.2%}")

    if final_model_name == "Logistic Regression":
        final_prob = lr_prob
        final_label = lr_label
    else:
        final_prob = rf_prob
        final_label = rf_label

    print(f"\nFinal ({final_model_name}) -> Churn: {final_label}, Probability: {final_prob:.2%}")

# Helper: same style, but the final decision is always based on Random Forest
def show_prediction_rf_final(title, raw_customer):
    print(f"\n{title}")
    print("-" * 80)
    customer_input = pd.DataFrame([raw_customer])
    print("New customer input:")
    print(customer_input)

    model_input = prepare_customer(raw_customer)
    lr_prob = log_reg.predict_proba(scaler.transform(model_input))[0, 1]
    rf_prob = random_forest.predict_proba(model_input)[0, 1]

    lr_label = "Yes" if lr_prob >= 0.5 else "No"
    rf_label = "Yes" if rf_prob >= 0.5 else "No"

    print("\nPrediction output:")
    print(f"Logistic Regression -> Churn: {lr_label}, Probability: {lr_prob:.2%}")
    print(f"Random Forest       -> Churn: {rf_label}, Probability: {rf_prob:.2%}")

    final_model_name_rf = "Random Forest"
    final_prob_rf = rf_prob
    final_label_rf = rf_label
    print(f"\nFinal ({final_model_name_rf}) -> Churn: {final_label_rf}, Probability: {final_prob_rf:.2%}")

# Example 1: likely to stay
new_customer_no = {
    "CLIENTNUM": 999999999,
    "Attrition_Flag": "Existing Customer",
    "Customer_Age": 52,
    "Gender": "M",
    "Dependent_count": 2,
    "Education_Level": "Graduate",
    "Marital_Status": "Married",
    "Income_Category": "$80K - $120K",
    "Card_Category": "Blue",
    "Months_on_book": 56,
    "Total_Relationship_Count": 4,
    "Months_Inactive_12_mon": 1,
    "Contacts_Count_12_mon": 1,
    "Credit_Limit": 6000,
    "Total_Revolving_Bal": 2000,
    "Avg_Open_To_Buy": 4000,
    "Total_Amt_Chng_Q4_Q1": 0.92,
    "Total_Trans_Amt": 2744,
    "Total_Trans_Ct": 42,
    "Total_Ct_Chng_Q4_Q1": 0.85,
    "Avg_Utilization_Ratio": 0.33,
}

# Example 2: likely to leave
new_customer_yes = {
    "CLIENTNUM": 888888888,
    "Attrition_Flag": "Attrited Customer",
    "Customer_Age": 39,
    "Gender": "F",
    "Dependent_count": 1,
    "Education_Level": "College",
    "Marital_Status": "Single",
    "Income_Category": "Less than $40K",
    "Card_Category": "Blue",
    "Months_on_book": 14,
    "Total_Relationship_Count": 1,
    "Months_Inactive_12_mon": 4,
    "Contacts_Count_12_mon": 4,
    "Credit_Limit": 3000,
    "Total_Revolving_Bal": 500,
    "Avg_Open_To_Buy": 2500,
    "Total_Amt_Chng_Q4_Q1": 0.45,
    "Total_Trans_Amt": 1232,
    "Total_Trans_Ct": 18,
    "Total_Ct_Chng_Q4_Q1": 0.30,
    "Avg_Utilization_Ratio": 0.79,
}

# Example 3: another customer likely to stay
new_customer_no_2 = {
    "CLIENTNUM": 777777777,
    "Attrition_Flag": "Existing Customer",
    "Customer_Age": 45,
    "Gender": "F",
    "Dependent_count": 3,
    "Education_Level": "High School",
    "Marital_Status": "Married",
    "Income_Category": "$40K - $60K",
    "Card_Category": "Blue",
    "Months_on_book": 48,
    "Total_Relationship_Count": 5,
    "Months_Inactive_12_mon": 0,
    "Contacts_Count_12_mon": 1,
    "Credit_Limit": 7500,
    "Total_Revolving_Bal": 3000,
    "Avg_Open_To_Buy": 4500,
    "Total_Amt_Chng_Q4_Q1": 1.10,
    "Total_Trans_Amt": 3600,
    "Total_Trans_Ct": 58,
    "Total_Ct_Chng_Q4_Q1": 0.95,
    "Avg_Utilization_Ratio": 0.28,
}

# Example 4: random forest final example tuned to show a No result
new_customer_rf_final = {
    "CLIENTNUM": 666666666,
    "Attrition_Flag": "Existing Customer",
    "Customer_Age": 58,
    "Gender": "M",
    "Dependent_count": 1,
    "Education_Level": "Graduate",
    "Marital_Status": "Married",
    "Income_Category": "$80K - $120K",
    "Card_Category": "Blue",
    "Months_on_book": 72,
    "Total_Relationship_Count": 6,
    "Months_Inactive_12_mon": 0,
    "Contacts_Count_12_mon": 0,
    "Credit_Limit": 12000,
    "Total_Revolving_Bal": 5000,
    "Avg_Open_To_Buy": 7000,
    "Total_Amt_Chng_Q4_Q1": 1.25,
    "Total_Trans_Amt": 5200,
    "Total_Trans_Ct": 78,
    "Total_Ct_Chng_Q4_Q1": 1.10,
    "Avg_Utilization_Ratio": 0.22,
}

show_prediction("Example 1: No churn test", new_customer_no)
show_prediction("Example 2: Churn test", new_customer_yes)
show_prediction("Example 3: Another No churn test", new_customer_no_2)
show_prediction_rf_final("Example 4: Random Forest final result", new_customer_rf_final)

RAW CUSTOMER EXAMPLES

Example 1: No churn test
--------------------------------------------------------------------------------
New customer input:
   CLIENTNUM     Attrition_Flag  Customer_Age Gender  Dependent_count  \
0  999999999  Existing Customer            52      M                2   

  Education_Level Marital_Status Income_Category Card_Category  \
0        Graduate        Married    $80K - $120K          Blue   

   Months_on_book  ...  Months_Inactive_12_mon  Contacts_Count_12_mon  \
0              56  ...                       1                      1   

   Credit_Limit  Total_Revolving_Bal  Avg_Open_To_Buy  Total_Amt_Chng_Q4_Q1  \
0          6000                 2000             4000                  0.92   

   Total_Trans_Amt  Total_Trans_Ct  Total_Ct_Chng_Q4_Q1  Avg_Utilization_Ratio  
0             2744              42                 0.85                   0.33  

[1 rows x 21 columns]

Prediction output:
Logistic Regression -> Churn: No, Probability: 33.81%
Random